In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats
from scipy.integrate import quad
from scipy.interpolate import UnivariateSpline
import seaborn as sns
from scipy.special import factorial
import math
from iminuit import cost, Minuit 
import numpy as np
import pickle

sns.set_context('talk', font_scale=1.0)
sns.set_palette('colorblind')

from depth_helpers import *

Cs_line = source_dict['Cs137']
Am_line = source_dict['Am241']

### Inputs match pixelbypixel_correction_postLL_2025.ipynb (Cs137, CTDs, sims) and Am241_mapping_HP41520_1.ipynb (post-LL1 Am241)
datpath = '/Users/Sophie/Desktop/COSI/calibrations/Data/CalibratedDATs/'
plot_dir = '/Users/Sophie/Desktop/COSI/calibrations/Data/trapping_figures/postLL/'

############ files contain gaussian mean of CTD value for illumination on AC and DC side respectively, for each pixel ##############
AC_illumination_CTD = '/Users/Sophie/Desktop/COSI/calibrations/Data/CTD_params/20262201_Am241_AC_1200V_80K_57keV_to_62keV/CTD_parameters.txt'
DC_illumination_CTD = '/Users/Sophie/Desktop/COSI/calibrations/Data/CTD_params/20262201_Am241_DC_1200V_80K_57keV_to_62keV/CTD_parameters.txt'

CTD_sims = '/Users/Sophie/Desktop/COSI/calibrations/Data/SimulatedCSVs/HP41520_1/CTD_HP415201.csv'
AC_sim_events = '/Users/Sophie/Desktop/COSI/calibrations/Data/SimulatedCSVs/HP41520_1/am241_interactions_HP415201_ac.csv'
DC_sim_events = '/Users/Sophie/Desktop/COSI/calibrations/Data/SimulatedCSVs/HP41520_1/am241_interactions_HP415201_dc.csv'
sim_dCCE = '/Users/Sophie/Desktop/COSI/calibrations/Data/SimulatedCSVs/HP41520_1/HP415201_dCCE_1000cm.csv'

calibrator = DepthCalibrator_Am241(AC_illumination_CTD, DC_illumination_CTD, AC_sim_events, DC_sim_events, CTD_sims)
sim_depth, sim_ctd = calibrator.get_simdata()

In [ ]:
### Only Cs137 and Am241 are corrected. Each source's dat files live in a different directory under datpath.
dat_dir_dict = {'Cs137': 'postLL_2025/recalib/', 'Am241': 'Am241_2025/'}

file_dict = {'Cs137': [['20251217_Cs137_AC_1200V_80K_1.dat', '20251217_Cs137_AC_1200V_80K.dat', '20251219_Cs137_AC_1200V_80K.dat', '20260121_Cs137_AC_1200V_80K.dat'], \
                       ['20260123_Cs137_DC_1200V_80K.dat']], \
             'Am241': [['20262201_Am241_AC_1200V_80K.dat'], \
                       ['20262201_Am241_DC_1200V_80K.dat']]}

### n strips excluded in pixelbypixel_correction_postLL_2025.ipynb
bad_n_strips = [28, 5, 25]

erange_dict = {'Cs137': [630., 667.], 'Am241': [50., 64.]}
zbin_dict = {'Cs137': np.linspace(calibrator.zmin, calibrator.zmax, 31), \
             'Am241': np.linspace(calibrator.zmin, calibrator.zmax, 31)}

# Loop and make depth-calibrated dataframes

In [ ]:
data_dict = {}
for source in file_dict:
    emin = erange_dict[source][0]
    emax = erange_dict[source][1]
    
    string = source + 'allACDC_postLL1'
    
    ## Read in the dat files
    fullpaths = []
    for p in np.concatenate(file_dict[source]):
        fullpaths.append(datpath + dat_dir_dict[source] + p)
    
    df = make_df_from_dat(fullpaths, e_min=emin, e_max=emax)
    df = df[~df.strip_n.isin(bad_n_strips)].copy()

    ### Run depth calibration

    ctd_obs_map = [[[] for p in range(37)] for n in range(37)]
    ctd_stretch_map = [[[] for p in range(37)] for n in range(37)]
    ctd_obs_flat = []
    ctd_stretch_flat = []

    for p in range(37):
        for n in range(37):
            ctd_obs, ctd_stretch, depth, depth_err, bad = calibrator.depth_from_timing_prob(p+1, n+1, df.loc[df.strip_p.eq(p+1)&df.strip_n.eq(n+1), 'time_p'], \
                                                                                       df.loc[df.strip_p.eq(p+1)&df.strip_n.eq(n+1), 'time_n'], \
                                                                                      df.loc[df.strip_p.eq(p+1)&df.strip_n.eq(n+1), 'energy_p'])
            df.loc[df.strip_p.eq(p+1)&df.strip_n.eq(n+1), 'z'] =  depth
            df.loc[df.strip_p.eq(p+1)&df.strip_n.eq(n+1), 'z_err'] =  depth_err
            df.loc[df.strip_p.eq(p+1)&df.strip_n.eq(n+1), 'bad'] =  bad
            ctd_obs_map[p][n] = ctd_obs[~bad]
            ctd_stretch_map[p][n] = ctd_stretch[~bad]
            ctd_obs_flat.append(ctd_obs[~bad])
            ctd_stretch_flat.append(ctd_stretch[~bad])

    ctd_obs_flat = np.concatenate(ctd_obs_flat)
    ctd_stretch_flat = np.concatenate(ctd_stretch_flat)
    df['bad'] = df['bad'].astype(bool)
    data_dict[source] = df
    
    plt.figure()
    plt.hist(df["z"], bins=50, histtype = "step", range=(calibrator.zmin,calibrator.zmax))
    plt.yscale('log')
    plt.xlabel("z (cm)")
    plt.ylabel("counts")
    plt.title(source)
    plt.tight_layout()
    plt.savefig(plot_dir + 'depth_' + string + '_prob.pdf')
    plt.show()

    plt.figure()
    plt.hist(ctd_obs_flat, bins=50, histtype="step", label="Observed CTD")
    plt.hist(ctd_stretch_flat, bins=50, histtype="step", color='red', label="stretched and shifted")
    plt.axvline(np.max(sim_ctd), lw=0.8, color='red', ls='dashed', zorder=0)
    plt.axvline(np.min(sim_ctd), lw=0.8, color='red', ls='dashed', zorder=0)
    plt.yscale('log')
    plt.xlabel('CTD')
    plt.ylabel('counts')
    plt.title(source)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir + "CTD_" + string + ".pdf")
    plt.show()

    ### Uncorrected photopeaks (good-depth events only)
    bins = int((emax-emin) * 4)
    for side in ['p', 'n']:
        plt.figure()
        hist,binedges,_  = plt.hist(df['energy_'+side][~df['bad'].values], bins=bins, range=(emin, emax))
        plt.axvline(source_dict[source], color='red', ls='dashed')
        plt.yscale('log')
        plt.xlabel('Energy (keV)')
        plt.ylabel('Counts')
        plt.title(source + ', ' + side + ' side (uncorrected)')
        plt.show()

        fwhm, fwtm, fwhm_err, fwtm_err = get_FWHM_FWTM(binedges[:-1], hist)
        print(source, side, 'FWHM = ' + str(round(fwhm, 2)), 'FWTM = ' + str(round(fwtm, 2)))

# Depth plots: photopeak centroid vs. depth

In [ ]:
depth_plot_dict = {}
for source in data_dict:
    string = source + 'allACDC_postLL1'
    z_bins, e_trapping, h_trapping = make_depthplot(data_dict[source], string, zbin_dict[source], plot_dir=plot_dir, source=source)
    depth_plot_dict[source] = z_bins, e_trapping, h_trapping
    with open('./depth_plot_'+string+'.txt', 'wb') as file:
        pickle.dump([z_bins, e_trapping, h_trapping], file)

# Fit Cs137 depth plot to the simulated CCE curve

Cs137 interacts throughout the detector, so its depth plot constrains the trapping parameters b (electrons) and c (holes) along with the normalizations ae, ah.

In [ ]:
trim_index = 1
string = 'Cs137allACDC_postLL1'
m_Cs = fit_CCE(*depth_plot_dict['Cs137'], sim_dCCE, plot_dir=plot_dir, plot_suffix=string, trim_index=trim_index, source='Cs137')
print(m_Cs)
b_Cs, c_Cs = m_Cs.values['b', 'c']

# Fit Am241 normalizations with the Cs137 trapping parameters

Am241 only populates the depths near each face, so it cannot constrain b and c on its own. Fix b and c to the Cs137 values and fit only ae and ah (absorbs the Am-line gain offset of each side). Empty or failed depth bins are skipped.

In [ ]:
sim_dCCE_dat = load_sim_dCCE(sim_dCCE)

def e_depth_plot(z, ae, b, c):
    CCE = ae*(1.-b*sim_dCCE_dat[1][::-1])*(1.-c*sim_dCCE_dat[2][::-1])
    return UnivariateSpline(sim_dCCE_dat[0], CCE)(z)

def h_depth_plot(z, ah, b, c):
    CCE = ah*(1.-b*sim_dCCE_dat[3][::-1])*(1.-c*sim_dCCE_dat[4][::-1])
    return UnivariateSpline(sim_dCCE_dat[0], CCE)(z)

z_bins, e_trapping, h_trapping = depth_plot_dict['Am241']
z_list = (z_bins[:-1] + z_bins[1:])/2.
z_err = (z_bins[1:]-z_bins[:-1])/2.

e_ok = np.isfinite(e_trapping[0]) & np.isfinite(e_trapping[1]) & (e_trapping[1] > 0)
h_ok = np.isfinite(h_trapping[0]) & np.isfinite(h_trapping[1]) & (h_trapping[1] > 0)

c_Am = cost.LeastSquares(z_list[e_ok], e_trapping[0][e_ok], e_trapping[1][e_ok], e_depth_plot) + \
       cost.LeastSquares(z_list[h_ok], h_trapping[0][h_ok], h_trapping[1][h_ok], h_depth_plot)
m_Am = Minuit(c_Am, ae=np.nanmax(e_trapping[0]), ah=np.nanmax(h_trapping[0]), b=b_Cs, c=c_Cs)
m_Am.fixed['b', 'c'] = True
m_Am.migrad()
m_Am.hesse()
print(m_Am)

plot_suffix = 'Am241allACDC_postLL1_Cs137params'
plt.figure()
plt.errorbar(z_list[e_ok], e_trapping[0][e_ok]*Am_line, xerr=z_err[e_ok], yerr=e_trapping[1][e_ok]*Am_line, fmt=".", label="HV signal")
plt.errorbar(z_list[h_ok], h_trapping[0][h_ok]*Am_line, xerr=z_err[h_ok], yerr=h_trapping[1][h_ok]*Am_line, fmt=".", label="LV signal")
plt.plot(sim_dCCE_dat[0], e_depth_plot(sim_dCCE_dat[0], *m_Am.values['ae', 'b', 'c'])*Am_line, color='C0', zorder=0, lw=0.9)
plt.plot(sim_dCCE_dat[0], h_depth_plot(sim_dCCE_dat[0], *m_Am.values['ah', 'b', 'c'])*Am_line, color='C1', zorder=0, lw=0.9)
plt.axhline(Am_line, ls='--', color='C2', zorder=0)
plt.legend()
plt.xlabel("Detector Depth (cm)"); plt.ylabel("Centroid Energy (keV)")
plt.tight_layout()
plt.savefig(plot_dir + 'CCE_fit_energy_' + plot_suffix + '.pdf')
plt.show()

# Correct Cs137 and Am241 photopeak energies with the depth-dependent CCE

In [ ]:
### Both sources use the trapping parameters (b, c) from the Cs137 fit, with their own normalizations (ae, ah).
cce_params = {'Cs137': m_Cs.values['ae', 'ah'], 'Am241': m_Am.values['ae', 'ah']}

for source in data_dict:
    string = source + 'allACDC_postLL1_Cs137params'
    print(source)
    depth_correction_CCE(data_dict[source], *cce_params[source], b_Cs, c_Cs, sim_dCCE, plot_dir=plot_dir, plot_suffix=string, source=source)

In [ ]:
##################### Uncorrected vs. corrected photopeaks #####################
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, source in enumerate(data_dict):
    df = data_dict[source]
    emin, emax = erange_dict[source]
    bins = np.linspace(emin, emax, int((emax - emin) * 4) + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    mask = ~df['bad'].values
    for j, side in enumerate(['p', 'n']):
        ax = axes[i][j]
        for col, label, color in [('energy_' + side, 'Uncorrected', 'C0'), ('depth_corrected_energy_' + side, 'Corrected', 'C1')]:
            E = df[col].values[mask]
            hist, _ = np.histogram(E[np.isfinite(E)], bins=bins)
            fwhm, fwtm, _, _ = get_FWHM_FWTM(centers, hist)
            ax.stairs(hist, bins, color=color, lw=1.5, label='%s: FWHM %.2f, FWTM %.2f keV' % (label, fwhm, fwtm))
        ax.axvline(source_dict[source], color='gray', ls=':')
        ax.set_yscale('log')
        ax.set_title('%s, %s side' % (source, side), fontsize='small')
        ax.set_xlabel('Energy (keV)')
        ax.set_ylabel('Counts')
        ax.legend(loc='upper left', fontsize='x-small', frameon=False)
plt.tight_layout()
plt.savefig(plot_dir + 'CCEdepth_corrected_CsAm_overlay_postLL1.pdf')
plt.show()